In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [10]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Veri Yolu
data_dir = "/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray"
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')
val_dir = os.path.join(data_dir, 'val')

# Görüntü Ayarları
IMG_SIZE = (224, 224)
BATCH_SIZE = 128

print("Veriler hazırlanıyor...")

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224), # Modelin beklediği boyut
    batch_size=32,
    class_mode='binary',
    shuffle=False # Test ederken karıştırma yapmıyoruz
)

# --- TRANSFER LEARNING (VGG16) ---
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Hazır modelin ağırlıklarını dondurma
for layer in base_model.layers:
    layer.trainable = False

# Yeni başlık
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Eğitim başlıyor.")

history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)
print("Eğitim sonra erdi.")

# Model Testi
loss, acc = model.evaluate(test_generator)

print(f"Test sonucu: %{acc*100:.2f}")

# Modeli Kaydetme
model.save("xray_model.h5")
print("Model hazır: xray_model.h5 ")

Veriler hazırlanıyor...
Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Eğitim başlıyor.
Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.7134 - loss: 0.5730 - val_accuracy: 0.6875 - val_loss: 0.5868
Epoch 2/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - accuracy: 0.8606 - loss: 0.3357 - val_accuracy: 0.7500 - val_loss: 0.5790
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - accuracy: 0.8941 - loss: 0.2706 - val_accuracy: 0.7500 - val_loss: 0.6294
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - accuracy: 0.9149 - loss: 0.2235 - val_accuracy: 0.7500 - val_loss: 0.7153
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - accuracy: 0.9117 - loss: 0.2361 - val_accuracy: 0.7500 - val_loss: 0.5788
Eğitim sonra erdi.
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 400ms/step - accuracy: 0.8165 - loss: 0.4036


Test sonucu: %86.70
Model hazır: xray_model.h5 
